In [2]:
import re
import os
from pathlib import Path
import numpy as np

def calc_score(dataset_name, split, fewshot_bool, fewshot_num, full_or_LP, loss_fun):
    base_dir       = Path("results")
    LOWER_IS_BETTER = ['lipo', 'esol', 'malaria', 'cep']

    # ─── Build glob‐style pattern ─────────────────────────────────────────────────
    pattern = (
        f"FT_{dataset_name}_{split}_Fewshot_{fewshot_bool}_"
        f"{fewshot_num}_{full_or_LP}_type_{loss_fun}_*"
    )
    candidate_dirs = list(base_dir.glob(pattern))
    # print(candidate_dirs)
    results = {}  # folder_name → (mean, std)
    for folder in candidate_dirs:
        if not folder.is_dir():
            continue

        # assume each folder contains subdirs for different runs/seeds
        subdirs = [d for d in folder.iterdir() if d.is_dir()]
        vals = []

        for sd in subdirs:
            log_path = sd / "logging.log"
            if not log_path.exists():
                continue

            lines = log_path.read_text().splitlines()
            # find all indices where the run finishes
            done_idxs = [i for i, l in enumerate(lines) if l.startswith("Done! took:")]
            for idx in done_idxs:
                if idx == 0:
                    continue
                prev_line = lines[idx - 1].strip()
                nums = re.findall(r"[-+]?\d*\.\d+|\d+", prev_line)
                if nums:
                    vals.append(float(nums[-1]))
        # print(vals)
        if vals:
            avg = float(np.mean(vals))
            sd  = float(np.std(vals, ddof=0))
            results[folder.name] = (avg, sd)
    
    # print(results)
    # ─── Select best hyperparameter folder ────────────────────────────────────────
    if not results:
        return None, None, None

    if dataset_name in LOWER_IS_BETTER:
        best_folder, (best_avg, best_sd) = min(results.items(), key=lambda kv: kv[1][0])
    else:
        best_folder, (best_avg, best_sd) = max(results.items(), key=lambda kv: kv[1][0])

    return best_folder, best_avg, best_sd

In [325]:
# dataset_name = 'esol'
['clintox', 'bbbp', 'bace', 'hiv', 'muv', 'sider', 'tox21', 'toxcast']
fewshot_bool = True
fewshot_num = 50
split = 'size'
# base_loss = 'mse' if dataset_name in ['lipo', 'esol', 'malaria', 'cep'] else 'filter_bce'
full_or_LP = ["DWISE"]
# for split in splits:
for dataset_name in ['clintox', 'bbbp', 'bace', 'hiv', 'muv', 'sider', 'tox21', 'toxcast']:
    base_loss = 'mse' if dataset_name in ['lipo', 'esol', 'malaria', 'cep'] else 'filter_bce'
    loss_funcs = [base_loss]
    for loss_fun in loss_funcs:
        for full_bool in full_or_LP:
            best_folder, best_avg, best_sd = calc_score(dataset_name, split, fewshot_bool, fewshot_num, full_bool, loss_fun)
            if best_folder:
                print("\nBest hyperparameter setting:")
                print(f"  → {best_folder}")
                if dataset_name not in ['lipo', 'esol', 'malaria', 'cep']:
                    # percentage form, ready to paste into one Google-Sheets cell
                    pct_mean = best_avg * 100
                    pct_std  = best_sd  * 100
                    formatted = f"{pct_mean:.2f} ± {pct_std:.2f}"
                else:
                    # plain decimal with three digits
                    formatted = f"{best_avg:.3f} ± {best_sd:.3f}"
                print(f"     {formatted}")


Best hyperparameter setting:
  → FT_bace_size_Fewshot_True_50_DWISE_type_filter_bce_alpha1.0
     61.46 ± 0.57

Best hyperparameter setting:
  → FT_sider_size_Fewshot_True_50_DWISE_type_filter_bce_alpha1.0
     85.39 ± 0.23
